<a href="https://colab.research.google.com/github/kamranshahid56-tech/insurance_premium_prediction_model/blob/main/insurance_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [2]:
df = pd.read_csv('/content/insurance_premium_dataset.csv')

In [3]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,44,89.3,1.65,13.505166,False,Kota,government_job,Medium
107,57,51.2,1.58,16.595240,False,Mysore,government_job,Low
63,21,79.4,1.78,1.114412,True,Hyderabad,student,Low
9,45,89.6,1.93,19.266983,False,Hyderabad,government_job,Low
53,80,83.6,1.58,0.915343,False,Hyderabad,retired,Medium


In [4]:
df['occupation'].unique()

array(['government_job', 'private_job', 'retired', 'student', 'business'],
      dtype=object)

In [5]:
df_feat = df.copy()

In [8]:
#feature 1 BMI
df_feat['bmi'] = df['weight']/(df['height']**2)

In [9]:
# Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

In [10]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [11]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [12]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [13]:
tier_1_cities = ["Bangalore", "Chennai", "Delhi", "Hyderabad", "Mumbai"]

tier_2_cities = ["Mysore"]

tier_3_cities = ["Kota"]

In [14]:
# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [15]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [16]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
66,21.610346,private_job,24.195918,adult,medium,1,Medium
37,12.494193,business,21.511427,senior,medium,1,High
65,43.445174,business,22.815711,adult,low,3,Low
36,1.618954,retired,31.714539,senior,medium,1,Medium
0,13.505166,government_job,32.800735,adult,medium,3,Medium


In [17]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [20]:
X.head()

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,32.800735,adult,medium,3,13.505166,government_job
1,19.669038,middle_aged,low,2,18.600000,private_job
2,44.044444,senior,medium,2,1.120000,retired
3,38.675103,adult,medium,1,28.467885,government_job
4,24.338934,senior,medium,3,2.160000,retired


In [21]:
y.head()

,insurance_premium_category
0,Medium
1,Medium
2,High
3,Low
4,High


In [22]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [23]:
preprocessor = ColumnTransformer(
    transformers = [
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [24]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [25]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [26]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.782608695652174

In [27]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
109,24.453125,adult,low,1,4.087157,private_job
2,44.044444,senior,medium,2,1.120000,retired
113,31.868085,young,medium,3,7.035008,private_job
38,34.970782,adult,medium,2,43.713345,government_job
35,30.541383,senior,high,3,24.603665,government_job


In [28]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)